# Full evaluation notebook (Vast.ai, single GPU) -- A10

Runs the *full* (unsampled) benchmarks against the `a10` SFT checkpoint on the rented GPU, same
method as `kaggle/kaggle_eval.ipynb` used for `d4`/`d6`:

- `scripts/chat_eval.py`: ARC-Easy, ARC-Challenge, MMLU, GSM8K, HumanEval.
- `scripts/eval_blimp.py`: BLiMP (Warstadt et al. 2020), 67 categories x 1000 pairs.

Run this in the browser Jupyter app on the same instance `vastai/run_a10.sh` trained on (Instance
Portal -> Jupyter). Reuses the checkpoint already on local disk from that run if present --
only pulls from Drive as a fallback (e.g. a fresh instance). Download this notebook when done
(File -> Download) and archive it under `vastai/runs/`, same convention as `kaggle/runs/`.

No credentials are stored in this file -- Cell 2 prompts for them interactively (`getpass`) only
if the local cache is missing, same policy as the rest of this repo (never commit secrets).

## Cell 0: clean up disk before doing anything else

`run_a10.sh` already hit a disk-full crash once (16GB filled by accumulated pretrain
checkpoints -- see RESEARCH_LOG.md 2026-08-11). This box is likely still carrying that debt:
build caches from the pretrain run, and `base_checkpoints/a10` (~627MB per saved step, several
steps kept) which this eval notebook doesn't even need -- only the SFT checkpoint matters here,
and the base checkpoints are already safe on Drive. Clear both before pulling anything else in.

In [ ]:
import os
import subprocess

print("Before cleanup:")
!df -h /

# Build caches from run_a10.sh -- safe to drop, nothing here is needed again.
!rm -rf ~/.cache/uv ~/.cache/pip
!rm -rf ~/.cargo/registry/cache ~/.cargo/registry/src
subprocess.run(["bash", "-lc", "apt-get clean 2>/dev/null || true"])

# Base (pretrain) checkpoints -- already synced to Drive by run_a10.sh, and this notebook only
# needs the SFT checkpoint, not these. Safe to drop entirely, not just prune to the latest step.
base_ckpt_dir = os.path.expanduser("~/nanochat_cache/base_checkpoints/a10")
if os.path.isdir(base_ckpt_dir):
    print(f"Removing {base_ckpt_dir} (already on Drive, not needed for eval)...")
    !rm -rf {base_ckpt_dir}

print("After cleanup:")
!df -h /

## Cell 1: repo + deps (reuses `~/repo` if `run_a10.sh` already set it up)

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = os.path.expanduser("~/repo")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml

print("Cell 1 done.")

## Cell 2: locate checkpoint -- local cache first, rclone (interactive credentials) as fallback

In [ ]:
import os

NANOCHAT_BASE_DIR = os.path.expanduser("~/nanochat_cache")
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR

tokenizer_ok = os.path.exists(os.path.join(NANOCHAT_BASE_DIR, "tokenizer", "tokenizer.pkl"))
a10_ckpt_dir = os.path.join(NANOCHAT_BASE_DIR, "chatsft_checkpoints", "a10")
a10_ok = os.path.isdir(a10_ckpt_dir) and any(f.startswith("model_") for f in os.listdir(a10_ckpt_dir))

if tokenizer_ok and a10_ok:
    print("Found tokenizer + a10 SFT checkpoint already on local disk (from run_a10.sh) -- skipping Drive pull.")
else:
    print("Local cache incomplete, pulling from Drive. Enter the same 4 credentials used for run_a10.sh:")
    from getpass import getpass
    client_id = getpass("GDRIVE_CLIENT_ID: ")
    client_secret = getpass("GDRIVE_CLIENT_SECRET: ")
    oauth_token = getpass("GDRIVE_OAUTH_TOKEN (the whole JSON blob): ")
    folder_id = getpass("GDRIVE_FOLDER_ID: ")

    rclone_conf_dir = os.path.expanduser("~/.config/rclone")
    os.makedirs(rclone_conf_dir, exist_ok=True)
    with open(os.path.join(rclone_conf_dir, "rclone.conf"), "w") as f:
        f.write(
            "[gdrive]\n"
            "type = drive\n"
            "scope = drive\n"
            f"client_id = {client_id}\n"
            f"client_secret = {client_secret}\n"
            f"token = {oauth_token}\n"
            f"root_folder_id = {folder_id}\n"
            "team_drive =\n"
        )
    del client_id, client_secret, oauth_token, folder_id

    !rclone copy gdrive:tokenizer {NANOCHAT_BASE_DIR}/tokenizer --checksum -v
    !rclone copy gdrive:chatsft_checkpoints/a10 {a10_ckpt_dir} --checksum -v

print("Checkpoint ready:")
!ls {a10_ckpt_dir}

## Cell 3: full chat_eval.py -- a10

In [ ]:
import os
os.chdir(os.path.expanduser("~/repo"))

# Full run, no -x limit. If this is taking too long, interrupt and rerun with e.g. `-x 200`
# (chat_eval.py's --max-problems flag) for a faster, still-informative sample.
!python3 -m scripts.chat_eval -i sft -g a10 2>&1 | tee ~/chat_eval_a10.log

## Cell 4: full BLiMP eval -- a10

In [ ]:
import os
os.chdir(os.path.expanduser("~/repo"))

# All 67 categories, 1000 pairs each, batched. If too slow, lower --max-pairs (e.g. 200).
!python3 -m scripts.eval_blimp -i sft -g a10 --batch-size 64 2>&1 | tee ~/blimp_a10.log